# Height-Weight Analysis and Linear Regression

CSCE 580 - Intro to AI — Exercise: Height vs Weight

This notebook guides you through exploring a height/weight dataset and fitting a simple linear regression model.


Prereqs:
- Python 3
- Packages: pyexcel-ods3, pandas, numpy, matplotlib, seaborn, scikit-learn

If needed, run the next cell to install them into your current environment.

In [ ]:
import sys, subprocess
pkgs = ["pyexcel-ods3","pandas","numpy","matplotlib","seaborn","scikit-learn"]
cmd = [sys.executable, "-m", "pip", "install", "-q"] + pkgs
print("Installing packages if missing:", " \n".join(pkgs))
subprocess.run(cmd, check=False)


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from pyexcel_ods3 import get_data
sns.set_theme()
%matplotlib inline


Configure file paths and column names.

- DATA_ODS: path to your .ods file containing the dataset
- Optionally set DATA_CSV if you have a CSV fallback
- Set HEIGHT_COL and WEIGHT_COL to match your column names

In [ ]:
DATA_ODS = "height_weight.ods"  # Update if different
DATA_CSV = None                 # e.g., "height_weight.csv" or leave as None
HEIGHT_COL = "Height"           # Update to match your file
WEIGHT_COL = "Weight"           # Update to match your file


In [ ]:
def load_from_ods(path_ods: Path) -> pd.DataFrame:
    data = get_data(str(path_ods))
    if not data:
        raise ValueError(f"No sheets found in {path_ods}")
    sheet_name = next(iter(data))
    rows = data[sheet_name]
    if not rows:
        raise ValueError(f"Sheet '{sheet_name}' is empty in {path_ods}")
    header, *body = rows
    # Ensure header is a list of strings
    header = [str(h) if h is not None else f"col_{i}" for i, h in enumerate(header)]
    # Pad or trim rows to header length
    fixed = [ (row + [None] * (len(header) - len(row)))[:len(header)] for row in body ]
    df = pd.DataFrame(fixed, columns=header)
    return df

p_ods = Path(DATA_ODS) if DATA_ODS else None
p_csv = Path(DATA_CSV) if DATA_CSV else None

if p_ods and p_ods.exists():
    df = load_from_ods(p_ods)
    source_used = str(p_ods)
elif p_csv and p_csv.exists():
    df = pd.read_csv(p_csv)
    source_used = str(p_csv)
else:
    raise FileNotFoundError("Please provide an existing DATA_ODS (.ods) or DATA_CSV (.csv) file.")

print("Loaded:", source_used)
print("Columns:", list(df.columns))
df.head()


In [ ]:
# Basic EDA
display(df.describe(include='all'))
print("Rows, Columns:", df.shape)


In [ ]:
# Clean and coerce types
df[HEIGHT_COL] = pd.to_numeric(df[HEIGHT_COL], errors='coerce')
df[WEIGHT_COL] = pd.to_numeric(df[WEIGHT_COL], errors='coerce')
before = len(df)
df = df.dropna(subset=[HEIGHT_COL, WEIGHT_COL])
after = len(df)
print(f"Dropped {before - after} rows with missing {HEIGHT_COL}/{WEIGHT_COL}")


In [ ]:
# Visualization: scatter and trend
plt.figure(figsize=(6,4))
sns.scatterplot(data=df, x=HEIGHT_COL, y=WEIGHT_COL, s=30, alpha=0.7)
sns.regplot(data=df, x=HEIGHT_COL, y=WEIGHT_COL, scatter=False, color='red')
plt.title("Height vs Weight")
plt.show()


In [ ]:
# Train/test split and linear regression
X = df[[HEIGHT_COL]].values
y = df[WEIGHT_COL].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)

print("Intercept:", model.intercept_)
print("Slope:", model.coef_[0])


In [ ]:
# Evaluation
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)
print({"MAE": mae, "RMSE": rmse, "R2": r2})


In [ ]:
# Plot regression line against data
plt.figure(figsize=(6,4))
sns.scatterplot(x=X_test.flatten(), y=y_test, s=30, alpha=0.7, label='Test data')
x_line = np.linspace(X_test.min(), X_test.max(), 100)
y_line = model.predict(x_line.reshape(-1,1))
plt.plot(x_line, y_line, color='red', label='Fit')
plt.xlabel(HEIGHT_COL)
plt.ylabel(WEIGHT_COL)
plt.title("Linear Regression Fit (Test Set)")
plt.legend()
plt.show()


Exercises

1) Try polynomial features for height (degree 2 or 3) and compare metrics.
2) Add any additional features available in your dataset (e.g., gender) and see the impact.
3) Perform k-fold cross-validation for more robust evaluation.
4) Identify and handle outliers; report how metrics change.
5) Save your trained model and write a small function to load and predict.


In [ ]:
# Your work area
# TODO: Implement experiments described above
pass


In [ ]:
# Optional: Save model
import pickle
MODEL_PATH = "linear_model.pkl"
with open(MODEL_PATH, "wb") as f:
    pickle.dump(model, f)
print("Saved:", MODEL_PATH)
